
# Deterministic arbitrage detection with bid--ask prices

This Colab notebook implements the **common-maturity deterministic arbitrage detector** from the paper:

**Deterministic Arbitrage, Localized Arbitrage Portfolios, and Arbitrage-Consistent Projection**.

The notebook replaces the older auxiliary-`z` worst-case-profit search by the primal linear programme in Section 4.2 of the paper. The important modelling choices are:

1. The detector uses the **zero-cost bounded primal search**:

   $$
   \max\; D_0 + \sum_i D_i + \eta M,
   $$

   where the payoff is constrained to be nonnegative at the finite breakpoint set

   $$
   \{0\}\cup\{\text{all traded call and put strikes}\}
   $$

   and the right-tail slope is constrained to be nonnegative.

2. The bid--ask extension is implemented in the executable way described in Section 6:

   - a long option/share is paid at the **ask**;
   - a short option/share is received at the **bid**;
   - buy and sell quantities are separate nonnegative variables;
   - the net payoff position is buy quantity minus sell quantity.

3. The cash account uses the paper's continuously compounded convention:

   $$
   b \mapsto b e^{rT}.
   $$

4. Option quantities are continuous by default, because the paper formulates a linear programme over real portfolio weights. Integer lots are a practical trading restriction, not part of the theoretical no-arbitrage test.

5. Calls and puts are treated as **primitive independent instruments**. The code does not require call--put matching by strike and does not eliminate puts through put--call parity.

The output is an explicit portfolio. If the optimal value is positive, the returned zero-cost portfolio has terminal profit nonnegative for every terminal stock price and strictly positive for at least one terminal stock price.



## Cell 1 -- Install required packages

Run this cell in Colab before running the implementation. The detector itself uses SciPy's `linprog`. Yahoo Finance support is optional and is used only by the interactive market-data workflow.


In [ ]:

%pip -q install yfinance prettytable scipy pandas matplotlib



## Cell 2 -- Full implementation

This cell defines all functions used by the notebook:

- quote cleaning and bid--ask perturbation;
- the paper-consistent deterministic arbitrage LP;
- output tables and plots;
- a small paper example;
- optional localized target-positive arbitrage construction.

The central function is:

```python
build_and_solve_paper_arbitrage_lp(...)
```

It solves the Section 4.2 primal detector with the Section 6 bid--ask execution convention.


In [ ]:

#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
Deterministic arbitrage detection with bid-ask prices.

This implementation follows the common-maturity framework of the paper
"Deterministic Arbitrage, Localized Arbitrage Portfolios, and
Arbitrage-Consistent Projection".

The code is intentionally verbose and heavily commented. The goal is not only
numerical output, but also an auditable translation of the mathematics into a
Colab-ready workflow.

Main theoretical objects implemented here
----------------------------------------
1. Common-maturity terminal stock state:
       x = S_T in R_+.

2. Portfolio payoff with cash, stock, calls, and puts:
       G(x) = b exp(rT) + a x
              + sum_i gamma_i (x - K_i^C)^+
              + sum_j delta_j (K_j^P - x)^+.

3. Zero-cost deterministic arbitrage:
       setup cost = 0,
       G(x) >= 0 for every x >= 0,
       G(x) > 0 for at least one x >= 0.

4. Finite LP certificate:
   Since G is continuous and piecewise affine, it is enough to check:
       G(0) >= 0,
       G(K) >= 0 at every traded strike K,
       right-tail slope >= 0.

   The paper's LP strengthens this into a search for positive margins:
       maximize D_0 + sum_K D_K + eta M
   subject to
       G(0) >= D_0,
       G(K) >= D_K,
       tail slope >= M,
       D_0, D_K, M >= 0.

5. Bid-ask execution:
   For every risky instrument W we use two nonnegative quantities:
       buy_W  >= 0, paid at ask;
       sell_W >= 0, received at bid.

   The net payoff exposure is buy_W - sell_W.
   The executable setup cost is buy_W * ask_W - sell_W * bid_W.

Important difference from the previous notebook
-----------------------------------------------
The old notebook minimized an auxiliary variable z. That detects portfolios
with a strictly positive uniform worst-case lower bound. The paper's definition
is more general: the payoff may be zero in some states and positive in others.
This file therefore maximizes breakpoint/tail margins as in the paper.
"""

from __future__ import annotations

from dataclasses import dataclass
from datetime import datetime, timezone
from typing import Dict, Iterable, List, Optional, Sequence, Tuple

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

try:
    from IPython.display import display
except Exception:
    def display(obj):
        print(obj)

try:
    import yfinance as yf
except Exception:
    yf = None

from scipy.optimize import linprog

try:
    from prettytable import PrettyTable
except Exception:
    PrettyTable = None


# =============================================================================
# Configuration
# =============================================================================

@dataclass
class UserConfig:
    """User-controlled inputs and solver bounds.

    The theoretical deterministic-arbitrage test is a zero-cost test. For that
    reason setup_cost_budget should normally remain 0.0. A nonzero value is
    allowed only for experimentation: the constraints then work with financed
    terminal profit G(x) - setup_cost_budget * exp(rT), not with raw wealth G(x).
    """

    ticker: str = "AAPL"
    expiration_date: str = ""
    transaction_cost_perturbation_percent: float = 0.0

    # If True, lastPrice is used as both raw bid and raw ask before any
    # artificial spread widening. This can be useful when Yahoo bid/ask fields
    # are missing or stale, but it is less faithful to executable quotes.
    use_last_price: bool = False

    # Annual continuously compounded risk-free rate, in percent.
    # Example: 4.0 means r = 0.04 and cash grows by exp(0.04*T).
    risk_free_rate_percent: float = 4.0

    # The paper's arbitrage search is zero cost. Keep this at 0 for the
    # mathematical arbitrage detector.
    setup_cost_budget: float = 0.0

    # Optional absolute quote perturbation in price units. This is added to asks
    # and subtracted from bids after the percentage perturbation.
    absolute_perturbation: float = 0.0

    # Synthetic stock spread around the downloaded spot price. Yahoo normally
    # gives recent close/spot data, not a synchronized stock bid/ask pair.
    # A value of 0.10 means a 0.10% full spread around spot before perturbation.
    stock_spread_percent: float = 0.0

    # Total option-position limits. These are normalization/liquidity bounds.
    # They are needed because any arbitrage can otherwise be scaled indefinitely.
    max_total_call_options_to_buy: float = 40.0
    max_total_call_options_to_sell: float = 40.0
    max_total_put_options_to_buy: float = 40.0
    max_total_put_options_to_sell: float = 40.0

    # Per-instrument position limits. Each option row is an independent traded
    # instrument. The paper allows real-valued positions, so these are floats.
    max_call_options_to_buy_per_contract: float = 2.0
    max_call_options_to_sell_per_contract: float = 2.0
    max_put_options_to_buy_per_contract: float = 2.0
    max_put_options_to_sell_per_contract: float = 2.0

    # Stock buy/sell limits. Separate buy/sell variables are used for bid-ask
    # execution, exactly as in the paper's Section 6 extension.
    max_shares_to_buy: float = 100.0
    max_shares_to_sell: float = 100.0

    # Bank-account bounds. Positive bank means depositing cash at time zero;
    # negative bank means borrowing cash at time zero.
    max_deposit_limit: float = 1_000_000.0
    max_borrow_limit: float = 1_000_000.0

    # Weight eta in the paper's objective D_0 + sum_i D_i + eta*M.
    # Any positive value is theoretically valid. It only affects which
    # arbitrage is selected when there are many.
    eta_tail_weight: float = 1.0

    # Numerical and plotting settings.
    solver_tolerance: float = 1e-8
    plot_points: int = 900
    high_price_multiplier: float = 3.0


# =============================================================================
# Small display helpers
# =============================================================================

def _make_table(field_names: Sequence[str]):
    """Create a PrettyTable when available; otherwise return None."""
    if PrettyTable is None:
        return None
    return PrettyTable(field_names)


def _as_float_array(values) -> np.ndarray:
    """Convert a pandas/numpy/list object to a finite float numpy array."""
    arr = np.asarray(values, dtype=float)
    if np.any(~np.isfinite(arr)):
        raise ValueError("Input contains non-finite numeric values.")
    return arr


# =============================================================================
# Expiration-date discovery and Yahoo data acquisition
# =============================================================================

def _require_yfinance() -> None:
    """Raise a clear error if yfinance is not available."""
    if yf is None:
        raise RuntimeError(
            "yfinance is not available. In Colab, run the install cell first: "
            "%pip -q install yfinance prettytable scipy pandas matplotlib"
        )


def available_expirations(ticker: str) -> List[str]:
    """Return all option expiration dates currently listed by Yahoo Finance."""
    _require_yfinance()
    clean_ticker = ticker.upper().strip()
    if not clean_ticker:
        raise ValueError("ticker must be a non-empty string.")

    stock = yf.Ticker(clean_ticker)
    dates = list(stock.options or [])
    if not dates:
        raise RuntimeError(
            f"No option expiration dates were returned for ticker {clean_ticker!r}."
        )
    return dates


def show_available_expiration_dates(ticker: str) -> List[str]:
    """Print a numbered table of available Yahoo option expirations."""
    dates = available_expirations(ticker)

    print("\n" + "=" * 78)
    print(f"Available expiration dates for {ticker.upper().strip()}")
    print("=" * 78)

    table = _make_table(["#", "Expiration date"])
    if table is not None:
        for i, d in enumerate(dates):
            table.add_row([i, d])
        print(table)
    else:
        display(pd.DataFrame({"#": range(len(dates)), "Expiration date": dates}))

    return dates


def validate_expiration_date(
    ticker: str,
    expiration_date: str,
    dates: Optional[Sequence[str]] = None,
) -> None:
    """Validate that expiration_date belongs to Yahoo's available dates."""
    if dates is None:
        dates = available_expirations(ticker)

    if expiration_date not in dates:
        preview = ", ".join(list(dates)[:20])
        more = " ..." if len(dates) > 20 else ""
        raise ValueError(
            f"Expiration date {expiration_date!r} is not available for "
            f"{ticker.upper().strip()}. Available dates include: {preview}{more}"
        )


def year_fraction_to_expiration(expiration_date: str) -> float:
    """Compute ACT/365 year fraction from current UTC time to expiration."""
    expiry = pd.to_datetime(expiration_date).to_pydatetime()
    if expiry.tzinfo is None:
        expiry = expiry.replace(tzinfo=timezone.utc)
    now = datetime.now(timezone.utc)
    seconds = max((expiry - now).total_seconds(), 0.0)
    return seconds / (365.0 * 24.0 * 3600.0)


def _clean_option_side(df: pd.DataFrame, side: str, use_last_price: bool) -> pd.DataFrame:
    """Clean one side of a Yahoo option chain.

    No call-put matching is performed. Each valid row remains an independent
    traded instrument. The only filtering is quote-quality filtering:
    finite strike, finite bid/ask source fields, strike > 0, ask > 0,
    bid >= 0, and ask >= bid.
    """
    if df is None or df.empty:
        return pd.DataFrame(
            columns=["type", "strike", "lastPrice", "bid", "ask", "raw_bid", "raw_ask"]
        )

    cols = ["strike", "lastPrice", "bid", "ask"]
    out = df.loc[:, [c for c in cols if c in df.columns]].copy()
    for c in cols:
        if c not in out.columns:
            out[c] = np.nan
        out[c] = pd.to_numeric(out[c], errors="coerce")

    out["type"] = side

    if use_last_price:
        out["raw_bid"] = out["lastPrice"]
        out["raw_ask"] = out["lastPrice"]
    else:
        out["raw_bid"] = out["bid"]
        out["raw_ask"] = out["ask"]

    out = out.dropna(subset=["strike", "raw_bid", "raw_ask"])
    out = out.query("strike > 0 and raw_ask > 0 and raw_bid >= 0 and raw_ask >= raw_bid")
    out = out.sort_values("strike").reset_index(drop=True)

    return out[["type", "strike", "lastPrice", "bid", "ask", "raw_bid", "raw_ask"]]


def fetch_option_chain_from_yahoo(
    ticker: str,
    expiration_date: str,
    use_last_price: bool = False,
    known_expirations: Optional[Sequence[str]] = None,
) -> Tuple[float, pd.DataFrame, pd.DataFrame]:
    """Download spot, calls, and puts from Yahoo Finance."""
    _require_yfinance()

    clean_ticker = ticker.upper().strip()
    validate_expiration_date(clean_ticker, expiration_date, known_expirations)

    stock = yf.Ticker(clean_ticker)

    hist = stock.history(period="5d")
    close = hist.get("Close", pd.Series(dtype=float)).dropna()
    if close.empty:
        raise RuntimeError(f"Unable to fetch a recent spot price for {clean_ticker}.")
    spot = float(close.iloc[-1])

    chain = stock.option_chain(expiration_date)
    calls = _clean_option_side(chain.calls, "call", use_last_price)
    puts = _clean_option_side(chain.puts, "put", use_last_price)

    if calls.empty and puts.empty:
        raise RuntimeError(
            f"Yahoo returned no valid call or put quotes for {clean_ticker} "
            f"at expiration {expiration_date}."
        )

    return spot, calls, puts


# =============================================================================
# Bid-ask perturbation
# =============================================================================

def apply_transaction_cost_perturbation(
    quotes: pd.DataFrame,
    perturbation_percent: float,
    absolute_perturbation: float = 0.0,
) -> pd.DataFrame:
    """Conservatively widen bid-ask spreads.

    The perturbation is deliberately asymmetric from the trader's viewpoint:
      * effective ask is increased;
      * effective bid is decreased but never below zero.

    This does not change the theory. It simply asks the arbitrage detector to
    survive additional transaction-cost stress.
    """
    if perturbation_percent < 0:
        raise ValueError("transaction_cost_perturbation_percent must be nonnegative.")
    if absolute_perturbation < 0:
        raise ValueError("absolute_perturbation must be nonnegative.")

    out = quotes.copy()
    pct = float(perturbation_percent) / 100.0

    out["eff_ask"] = out["raw_ask"] * (1.0 + pct) + float(absolute_perturbation)
    out["eff_bid"] = np.maximum(
        0.0,
        out["raw_bid"] * (1.0 - pct) - float(absolute_perturbation),
    )

    out = out.query("eff_ask > 0 and eff_bid >= 0 and eff_ask >= eff_bid").copy()
    out = out.reset_index(drop=True)
    return out


def effective_stock_bid_ask(
    spot: float,
    stock_spread_percent: float,
    perturbation_percent: float,
) -> Tuple[float, float]:
    """Construct an effective stock bid/ask pair from a spot price."""
    if spot <= 0:
        raise ValueError("spot must be positive.")
    if stock_spread_percent < 0:
        raise ValueError("stock_spread_percent must be nonnegative.")
    if perturbation_percent < 0:
        raise ValueError("transaction_cost_perturbation_percent must be nonnegative.")

    half = float(stock_spread_percent) / 200.0
    pct = float(perturbation_percent) / 100.0

    base_bid = float(spot) * (1.0 - half)
    base_ask = float(spot) * (1.0 + half)

    eff_bid = max(0.0, base_bid * (1.0 - pct))
    eff_ask = base_ask * (1.0 + pct)

    if eff_ask < eff_bid:
        raise RuntimeError("Effective stock ask became smaller than effective stock bid.")
    return eff_bid, eff_ask


def _ensure_effective_quote_columns(quotes: pd.DataFrame, side: str) -> pd.DataFrame:
    """Ensure that a custom quote table has the required columns."""
    if quotes is None or len(quotes) == 0:
        return pd.DataFrame(columns=["type", "strike", "eff_bid", "eff_ask", "raw_bid", "raw_ask"])

    out = quotes.copy()
    if "strike" not in out.columns:
        raise ValueError(f"{side} quotes must contain a 'strike' column.")

    # Accept either effective bid/ask or raw bid/ask. If only raw columns are
    # present, use them as the effective executable prices.
    if "eff_bid" not in out.columns:
        if "raw_bid" in out.columns:
            out["eff_bid"] = out["raw_bid"]
        elif "bid" in out.columns:
            out["eff_bid"] = out["bid"]
        else:
            raise ValueError(f"{side} quotes need eff_bid, raw_bid, or bid.")

    if "eff_ask" not in out.columns:
        if "raw_ask" in out.columns:
            out["eff_ask"] = out["raw_ask"]
        elif "ask" in out.columns:
            out["eff_ask"] = out["ask"]
        else:
            raise ValueError(f"{side} quotes need eff_ask, raw_ask, or ask.")

    if "raw_bid" not in out.columns:
        out["raw_bid"] = out["eff_bid"]
    if "raw_ask" not in out.columns:
        out["raw_ask"] = out["eff_ask"]
    if "type" not in out.columns:
        out["type"] = side

    for c in ["strike", "eff_bid", "eff_ask", "raw_bid", "raw_ask"]:
        out[c] = pd.to_numeric(out[c], errors="coerce")

    out = out.dropna(subset=["strike", "eff_bid", "eff_ask"])
    out = out.query("strike > 0 and eff_ask > 0 and eff_bid >= 0 and eff_ask >= eff_bid")
    out = out.sort_values("strike").reset_index(drop=True)
    return out


# =============================================================================
# LP variable layout and payoff rows
# =============================================================================

def _variable_layout(num_calls: int, num_puts: int, num_certificate_points: int) -> Dict[str, int]:
    """Return indices for the LP variables.

    Portfolio variables:
      0                           ... m-1          call_buy
      m                           ... 2m-1         call_sell
      2m                          ... 2m+n-1       put_buy
      2m+n                        ... 2m+2n-1      put_sell
      2m+2n                                         share_buy
      2m+2n+1                                       share_sell
      2m+2n+2                                       bank

    Margin variables:
      D_start ... D_start+q-1                        D at 0 and strikes
      M_idx                                             tail-slope margin
    """
    m = int(num_calls)
    n = int(num_puts)
    q = int(num_certificate_points)

    share_buy_idx = 2 * m + 2 * n
    share_sell_idx = share_buy_idx + 1
    bank_idx = share_sell_idx + 1
    n_portfolio = bank_idx + 1
    D_start = n_portfolio
    M_idx = D_start + q
    n_variables = M_idx + 1

    return {
        "m": m,
        "n": n,
        "share_buy_idx": share_buy_idx,
        "share_sell_idx": share_sell_idx,
        "bank_idx": bank_idx,
        "n_portfolio": n_portfolio,
        "D_start": D_start,
        "M_idx": M_idx,
        "n_variables": n_variables,
    }


def _portfolio_payoff_coefficients(
    x_value: float,
    call_strikes: np.ndarray,
    put_strikes: np.ndarray,
    bank_growth: float,
) -> np.ndarray:
    """Return coefficients of terminal wealth G(x) for portfolio variables.

    This row deliberately excludes the D and M margin variables. It corresponds
    only to the real traded portfolio:
        call_buy, call_sell, put_buy, put_sell, share_buy, share_sell, bank.
    """
    x_value = float(x_value)
    call_pay = np.maximum(x_value - call_strikes, 0.0)
    put_pay = np.maximum(put_strikes - x_value, 0.0)

    return np.concatenate([
        call_pay,                         # buying calls gives +call payoff
        -call_pay,                        # selling calls gives -call payoff
        put_pay,                          # buying puts gives +put payoff
        -put_pay,                         # selling puts gives -put payoff
        np.array([x_value], dtype=float), # buying shares gives +x
        np.array([-x_value], dtype=float),# selling shares gives -x
        np.array([bank_growth], dtype=float),
    ])


def _tail_slope_coefficients(num_calls: int, num_puts: int) -> np.ndarray:
    """Return coefficients of the right-tail slope for portfolio variables.

    For x beyond the largest strike:
      * each net call contributes slope +1;
      * puts contribute slope 0;
      * each net share contributes slope +1.

    Therefore slope = net shares + sum(net call positions).
    """
    m = int(num_calls)
    n = int(num_puts)
    layout = _variable_layout(m, n, 0)
    coeff = np.zeros(layout["n_portfolio"], dtype=float)

    coeff[0:m] = 1.0                  # call buys
    coeff[m:2*m] = -1.0               # call sells
    coeff[layout["share_buy_idx"]] = 1.0
    coeff[layout["share_sell_idx"]] = -1.0
    return coeff


def _initial_cost_coefficients(
    call_ask: np.ndarray,
    call_bid: np.ndarray,
    put_ask: np.ndarray,
    put_bid: np.ndarray,
    stock_ask: float,
    stock_bid: float,
) -> np.ndarray:
    """Return executable setup-cost coefficients for portfolio variables.

    Bid-ask convention:
      buy at ask  -> positive setup cost;
      sell at bid -> negative setup cost, because the trader receives cash.
    """
    m = len(call_ask)
    n = len(put_ask)
    layout = _variable_layout(m, n, 0)
    coeff = np.zeros(layout["n_portfolio"], dtype=float)

    coeff[0:m] = call_ask
    coeff[m:2*m] = -call_bid
    coeff[2*m:2*m+n] = put_ask
    coeff[2*m+n:2*m+2*n] = -put_bid
    coeff[layout["share_buy_idx"]] = float(stock_ask)
    coeff[layout["share_sell_idx"]] = -float(stock_bid)
    coeff[layout["bank_idx"]] = 1.0
    return coeff


# =============================================================================
# Core paper LP: zero-cost bounded primal search with bid-ask execution
# =============================================================================

def build_and_solve_paper_arbitrage_lp(
    spot: float,
    stock_bid: float,
    stock_ask: float,
    calls: pd.DataFrame,
    puts: pd.DataFrame,
    config: UserConfig,
    T: float,
) -> Dict[str, object]:
    """Build and solve the paper-consistent deterministic arbitrage LP.

    This implements the common-maturity primal search of Section 4.2, with the
    bid-ask execution convention of Section 6.

    Mathematically, the LP searches over a zero-cost bounded portfolio. It
    maximizes nonnegative margins at all payoff breakpoints and in the tail:

        maximize D_0 + sum_K D_K + eta M.

    If the optimal value is positive, the returned portfolio is a deterministic
    arbitrage under the paper's definition.
    """
    calls = _ensure_effective_quote_columns(calls, "call")
    puts = _ensure_effective_quote_columns(puts, "put")

    call_strikes = _as_float_array(calls["strike"])
    put_strikes = _as_float_array(puts["strike"])
    call_ask = _as_float_array(calls["eff_ask"])
    call_bid = _as_float_array(calls["eff_bid"])
    put_ask = _as_float_array(puts["eff_ask"])
    put_bid = _as_float_array(puts["eff_bid"])

    m = len(call_strikes)
    n = len(put_strikes)
    if m == 0 and n == 0:
        raise RuntimeError("No option instruments are available after quote filtering.")

    if stock_bid < 0 or stock_ask <= 0 or stock_bid > stock_ask:
        raise ValueError("Stock bid/ask must satisfy 0 <= bid <= ask and ask > 0.")

    # Paper convention: continuously compounded cash account.
    r = float(config.risk_free_rate_percent) / 100.0
    bank_growth = float(np.exp(r * float(T)))

    # The payoff is piecewise affine with breakpoints at all traded strikes.
    # Since calls and puts may be listed at different strikes, use the union.
    all_strikes = np.unique(np.concatenate([call_strikes, put_strikes]))
    certificate_points = np.unique(np.concatenate([np.array([0.0]), all_strikes]))
    q = len(certificate_points)

    layout = _variable_layout(m, n, q)
    L = layout["n_variables"]
    n_portfolio = layout["n_portfolio"]
    D_start = layout["D_start"]
    M_idx = layout["M_idx"]

    setup_budget = float(config.setup_cost_budget)

    # -----------------------------
    # Objective
    # -----------------------------
    # scipy.optimize.linprog minimizes. To maximize the paper objective, we
    # minimize its negative.
    c = np.zeros(L, dtype=float)
    c[D_start:D_start+q] = -1.0
    c[M_idx] = -float(config.eta_tail_weight)

    # -----------------------------
    # Equality constraints
    # -----------------------------
    # Executable setup cost equals the chosen budget. For deterministic
    # arbitrage this budget is zero.
    cost_coeff = _initial_cost_coefficients(
        call_ask=call_ask,
        call_bid=call_bid,
        put_ask=put_ask,
        put_bid=put_bid,
        stock_ask=stock_ask,
        stock_bid=stock_bid,
    )
    A_eq = np.zeros((1, L), dtype=float)
    A_eq[0, :n_portfolio] = cost_coeff
    b_eq = np.array([setup_budget], dtype=float)

    # -----------------------------
    # Inequality constraints A_ub x <= b_ub
    # -----------------------------
    rows = []
    rhs = []

    # Payoff-margin constraints:
    #     Pi(y) = G(y) - setup_budget * exp(rT) >= D_y.
    # Equivalently:
    #     -G(y) + D_y <= -setup_budget * exp(rT).
    for j, y in enumerate(certificate_points):
        row = np.zeros(L, dtype=float)
        row[:n_portfolio] = -_portfolio_payoff_coefficients(
            y,
            call_strikes,
            put_strikes,
            bank_growth,
        )
        row[D_start + j] = 1.0
        rows.append(row)
        rhs.append(-setup_budget * bank_growth)

    # Tail-slope margin:
    #     slope >= M
    # Equivalently:
    #     -slope + M <= 0.
    slope_coeff = _tail_slope_coefficients(m, n)
    row = np.zeros(L, dtype=float)
    row[:n_portfolio] = -slope_coeff
    row[M_idx] = 1.0
    rows.append(row)
    rhs.append(0.0)

    # Total quantity limits. These are not no-arbitrage assumptions; they are
    # normalization/liquidity bounds that make the arbitrage search bounded.
    row = np.zeros(L, dtype=float)
    row[0:m] = 1.0
    rows.append(row)
    rhs.append(float(config.max_total_call_options_to_buy))

    row = np.zeros(L, dtype=float)
    row[m:2*m] = 1.0
    rows.append(row)
    rhs.append(float(config.max_total_call_options_to_sell))

    row = np.zeros(L, dtype=float)
    row[2*m:2*m+n] = 1.0
    rows.append(row)
    rhs.append(float(config.max_total_put_options_to_buy))

    row = np.zeros(L, dtype=float)
    row[2*m+n:2*m+2*n] = 1.0
    rows.append(row)
    rhs.append(float(config.max_total_put_options_to_sell))

    A_ub = np.vstack(rows)
    b_ub = np.asarray(rhs, dtype=float)

    # -----------------------------
    # Variable bounds
    # -----------------------------
    bounds: List[Tuple[float, Optional[float]]] = []

    # Option buy/sell variables are nonnegative and bounded per instrument.
    bounds += [(0.0, float(config.max_call_options_to_buy_per_contract)) for _ in range(m)]
    bounds += [(0.0, float(config.max_call_options_to_sell_per_contract)) for _ in range(m)]
    bounds += [(0.0, float(config.max_put_options_to_buy_per_contract)) for _ in range(n)]
    bounds += [(0.0, float(config.max_put_options_to_sell_per_contract)) for _ in range(n)]

    # Stock buy/sell variables.
    bounds.append((0.0, float(config.max_shares_to_buy)))
    bounds.append((0.0, float(config.max_shares_to_sell)))

    # Bank account variable is free within borrowing/deposit bounds.
    bounds.append((-float(config.max_borrow_limit), float(config.max_deposit_limit)))

    # Breakpoint margins D_y and tail margin M are nonnegative.
    bounds += [(0.0, None) for _ in range(q)]
    bounds.append((0.0, None))

    # Solve the continuous LP. This is the paper's mathematical formulation.
    res = linprog(
        c=c,
        A_ub=A_ub,
        b_ub=b_ub,
        A_eq=A_eq,
        b_eq=b_eq,
        bounds=bounds,
        method="highs",
    )

    if not res.success or res.x is None:
        raise RuntimeError(
            "The LP solver did not return an optimal solution. "
            f"Status={res.status}, message={res.message}"
        )

    x = np.asarray(res.x, dtype=float)
    D_values = x[D_start:D_start+q]
    M_value = float(x[M_idx])
    objective_value = float(-res.fun)

    # For plotting, choose a finite upper endpoint. This is not used by the LP.
    max_reference = max(float(spot), float(certificate_points.max()), 1.0)
    S_high = float(config.high_price_multiplier) * max_reference

    result = {
        "solution": x,
        "status": res.status,
        "message": res.message,
        "objective_value": objective_value,
        "D_values": D_values,
        "M_value": M_value,
        "certificate_points": certificate_points,
        "all_strikes": all_strikes,
        "call_strikes": call_strikes,
        "put_strikes": put_strikes,
        "call_ask": call_ask,
        "call_bid": call_bid,
        "put_ask": put_ask,
        "put_bid": put_bid,
        "calls_table": calls,
        "puts_table": puts,
        "stock_bid": float(stock_bid),
        "stock_ask": float(stock_ask),
        "spot": float(spot),
        "T": float(T),
        "risk_free_rate": r,
        "bank_growth": bank_growth,
        "S_high": S_high,
        "layout": layout,
        "setup_budget": setup_budget,
        "config": config,
    }

    return result


# =============================================================================
# Evaluation and verification helpers
# =============================================================================

def portfolio_terminal_wealth(x_value: float, result: Dict[str, object]) -> float:
    """Evaluate raw terminal wealth G(x) of the selected portfolio."""
    layout = result["layout"]
    coeff = _portfolio_payoff_coefficients(
        x_value,
        result["call_strikes"],
        result["put_strikes"],
        result["bank_growth"],
    )
    portfolio_vars = result["solution"][:layout["n_portfolio"]]
    return float(np.dot(coeff, portfolio_vars))


def portfolio_terminal_profit(x_value: float, result: Dict[str, object]) -> float:
    """Evaluate financed terminal profit Pi(x) = G(x) - Y exp(rT)."""
    return float(
        portfolio_terminal_wealth(x_value, result)
        - float(result.get("setup_budget", 0.0)) * float(result["bank_growth"])
    )


def portfolio_tail_slope(result: Dict[str, object]) -> float:
    """Compute the actual right-tail slope of the selected portfolio."""
    layout = result["layout"]
    coeff = _tail_slope_coefficients(len(result["call_strikes"]), len(result["put_strikes"]))
    portfolio_vars = result["solution"][:layout["n_portfolio"]]
    return float(np.dot(coeff, portfolio_vars))


def executable_setup_cost(result: Dict[str, object]) -> float:
    """Compute executable initial cost using bid for sells and ask for buys."""
    layout = result["layout"]
    coeff = _initial_cost_coefficients(
        call_ask=result["call_ask"],
        call_bid=result["call_bid"],
        put_ask=result["put_ask"],
        put_bid=result["put_bid"],
        stock_ask=result["stock_ask"],
        stock_bid=result["stock_bid"],
    )
    portfolio_vars = result["solution"][:layout["n_portfolio"]]
    return float(np.dot(coeff, portfolio_vars))


def verify_finite_certificate(result: Dict[str, object], tolerance: Optional[float] = None) -> pd.DataFrame:
    """Return a finite certificate table for nonnegativity.

    By the paper's Lemma 4.1 logic, nonnegativity at these points plus a
    nonnegative tail slope certifies nonnegativity for every x >= 0.
    """
    if tolerance is None:
        tolerance = float(result["config"].solver_tolerance)

    rows = []
    D_values = result["D_values"]
    for y, d in zip(result["certificate_points"], D_values):
        pi_y = portfolio_terminal_profit(float(y), result)
        rows.append({
            "certificate_point": float(y),
            "profit_Pi": pi_y,
            "margin_D": float(d),
            "Pi_minus_D": pi_y - float(d),
            "passes": bool(pi_y + tolerance >= float(d)),
        })

    tail = portfolio_tail_slope(result)
    rows.append({
        "certificate_point": "tail_slope",
        "profit_Pi": tail,
        "margin_D": float(result["M_value"]),
        "Pi_minus_D": tail - float(result["M_value"]),
        "passes": bool(tail + tolerance >= float(result["M_value"])),
    })

    return pd.DataFrame(rows)


# =============================================================================
# Reporting
# =============================================================================

def print_quote_summary(
    spot: float,
    stock_bid: float,
    stock_ask: float,
    calls: pd.DataFrame,
    puts: pd.DataFrame,
    config: UserConfig,
    T: float,
    title: str = "Input summary",
) -> None:
    """Print a compact summary of the input quotes used by the model."""
    print("\n" + "=" * 78)
    print(title)
    print("=" * 78)
    print(f"Ticker / label: {config.ticker.upper().strip()}")
    print(f"Expiration date: {config.expiration_date}")
    print(f"Time to expiry T: {T:.8f} years")
    print(f"Spot: {spot:.8f}")
    print(f"Effective stock bid/ask: {stock_bid:.8f} / {stock_ask:.8f}")
    print(f"Continuously compounded risk-free rate: {config.risk_free_rate_percent:.8f}%")
    print(f"Cash growth exp(rT): {np.exp((config.risk_free_rate_percent/100.0)*T):.8f}")
    print(f"Setup-cost budget Y: {config.setup_cost_budget:.8f}")
    print(f"Transaction-cost perturbation: {config.transaction_cost_perturbation_percent:.8f}%")
    print(f"Absolute perturbation: {config.absolute_perturbation:.8f}")
    print(f"Calls used: {len(calls)}")
    print(f"Puts used: {len(puts)}")
    print("Calls and puts are used independently. No common-strike matching is performed.")


def print_portfolio_tables(result: Dict[str, object]) -> None:
    """Print stock, bank, option positions, and executable cashflow."""
    config = result["config"]
    x = result["solution"]
    layout = result["layout"]
    m = len(result["call_strikes"])
    n = len(result["put_strikes"])

    share_buy_idx = layout["share_buy_idx"]
    share_sell_idx = layout["share_sell_idx"]
    bank_idx = layout["bank_idx"]

    share_buy = float(x[share_buy_idx])
    share_sell = float(x[share_sell_idx])
    net_shares = share_buy - share_sell
    stock_cashflow = share_buy * float(result["stock_ask"]) - share_sell * float(result["stock_bid"])
    bank = float(x[bank_idx])

    print("\n" + "=" * 78)
    print("Portfolio table")
    print("=" * 78)

    rows = [
        {
            "Asset": f"{config.ticker.upper().strip()} stock",
            "Buy qty": share_buy,
            "Sell qty": share_sell,
            "Net qty": net_shares,
            "Executable prices": f"buy@ask={result['stock_ask']:.8f}, sell@bid={result['stock_bid']:.8f}",
            "Initial cashflow": stock_cashflow,
        },
        {
            "Asset": "Bank account",
            "Buy qty": np.nan,
            "Sell qty": np.nan,
            "Net qty": bank,
            "Executable prices": f"growth exp(rT)={result['bank_growth']:.8f}",
            "Initial cashflow": bank,
        },
    ]

    table = _make_table(["Asset", "Buy qty", "Sell qty", "Net qty", "Executable prices", "Initial cashflow"])
    if table is not None:
        for row in rows:
            table.add_row([
                row["Asset"],
                "" if pd.isna(row["Buy qty"]) else round(float(row["Buy qty"]), 8),
                "" if pd.isna(row["Sell qty"]) else round(float(row["Sell qty"]), 8),
                round(float(row["Net qty"]), 8),
                row["Executable prices"],
                round(float(row["Initial cashflow"]), 8),
            ])
        print(table)
    else:
        display(pd.DataFrame(rows))

    print("\n" + "=" * 78)
    print("Option-position table")
    print("=" * 78)

    option_rows = []
    tol = float(config.solver_tolerance)

    for i in range(m):
        qty = float(x[i])
        if qty > tol:
            option_rows.append({
                "Type": "Call buy",
                "Strike": float(result["call_strikes"][i]),
                "Qty": qty,
                "Price": float(result["call_ask"][i]),
                "Initial cashflow": qty * float(result["call_ask"][i]),
            })

    for i in range(m):
        idx = m + i
        qty = float(x[idx])
        if qty > tol:
            option_rows.append({
                "Type": "Call sell",
                "Strike": float(result["call_strikes"][i]),
                "Qty": qty,
                "Price": float(result["call_bid"][i]),
                "Initial cashflow": -qty * float(result["call_bid"][i]),
            })

    for i in range(n):
        idx = 2 * m + i
        qty = float(x[idx])
        if qty > tol:
            option_rows.append({
                "Type": "Put buy",
                "Strike": float(result["put_strikes"][i]),
                "Qty": qty,
                "Price": float(result["put_ask"][i]),
                "Initial cashflow": qty * float(result["put_ask"][i]),
            })

    for i in range(n):
        idx = 2 * m + n + i
        qty = float(x[idx])
        if qty > tol:
            option_rows.append({
                "Type": "Put sell",
                "Strike": float(result["put_strikes"][i]),
                "Qty": qty,
                "Price": float(result["put_bid"][i]),
                "Initial cashflow": -qty * float(result["put_bid"][i]),
            })

    if option_rows:
        option_df = pd.DataFrame(option_rows)
        if PrettyTable is not None:
            table = PrettyTable(["Type", "Strike", "Qty", "Price", "Initial cashflow"])
            for row in option_rows:
                table.add_row([
                    row["Type"],
                    round(row["Strike"], 8),
                    round(row["Qty"], 8),
                    round(row["Price"], 8),
                    round(row["Initial cashflow"], 8),
                ])
            table.add_divider()
            table.add_row([
                "Total amount",
                "--",
                "--",
                "--",
                round(float(option_df["Initial cashflow"].sum()), 8),
            ])
            print(table)
        else:
            display(option_df)
            print("Total option cashflow:", option_df["Initial cashflow"].sum())
    else:
        print("No option positions selected above numerical tolerance.")

    total_cost = executable_setup_cost(result)
    print(f"\nExecutable setup cost: {total_cost:.10f}")
    print(f"Required setup budget:  {float(result['setup_budget']):.10f}")
    print("Positive cashflow means cash paid by the trader; negative means cash received.")


def print_margin_table(result: Dict[str, object], max_rows: int = 30) -> None:
    """Print the breakpoint/tail margins used by the paper LP."""
    print("\n" + "=" * 78)
    print("Paper LP finite certificate margins")
    print("=" * 78)

    df = verify_finite_certificate(result)
    if len(df) > max_rows:
        print(f"Showing first {max_rows} rows out of {len(df)} certificate rows.")
        display(df.head(max_rows))
    else:
        display(df)


def print_conclusion(result: Dict[str, object]) -> None:
    """Print the deterministic-arbitrage conclusion."""
    config = result["config"]
    tol = float(config.solver_tolerance)
    W = float(result["objective_value"])
    budget = float(result.get("setup_budget", 0.0))
    min_profit = min(
        portfolio_terminal_profit(float(y), result)
        for y in result["certificate_points"]
    )
    tail_slope = portfolio_tail_slope(result)
    positive_breakpoint_margins = int(np.sum(result["D_values"] > tol))
    positive_tail_margin = bool(result["M_value"] > tol)

    print("\n" + "=" * 78)
    print("Arbitrage conclusion")
    print("=" * 78)
    print(f"LP status: {result['status']} -- {result['message']}")
    print(f"Optimal paper objective W*: {W:.10f}")
    print(f"Minimum certified finite profit: {min_profit:.10f}")
    print(f"Right-tail slope: {tail_slope:.10f}")
    print(f"Positive breakpoint margins: {positive_breakpoint_margins}")
    print(f"Positive tail margin: {positive_tail_margin}")

    if abs(budget) > tol:
        print("\n[NOTE] setup_cost_budget is not zero.")
        print("The result is a financed payoff search, not the pure zero-cost arbitrage test.")

    if W > tol and abs(budget) <= tol:
        print("\n[CONCLUSION] Deterministic arbitrage found.")
        print("The portfolio has zero executable setup cost, nonnegative terminal profit for all S_T >= 0,")
        print("and strictly positive terminal profit at some breakpoint or eventually in the right tail.")
    elif W > tol:
        print("\n[CONCLUSION] Positive financed payoff margins found for the chosen nonzero setup budget.")
        print("Set setup_cost_budget=0.0 to run the exact paper arbitrage test.")
    else:
        print("\n[CONCLUSION] No deterministic arbitrage was found within the imposed position bounds.")
        print("The optimal margins are zero up to numerical tolerance.")


def plot_profit_function(result: Dict[str, object]) -> None:
    """Plot terminal financed profit Pi(x) over a finite display range."""
    config = result["config"]
    y_max = max(float(result["S_high"]), float(result["spot"]) * 1.5, 1.0)
    y_vals = np.linspace(0.0, y_max, int(config.plot_points))
    profits = np.array([portfolio_terminal_profit(y, result) for y in y_vals])

    fig = plt.figure(figsize=(10, 6))
    ax = fig.gca()
    ax.plot(y_vals, profits, label="terminal financed profit Pi(S_T)")
    ax.axhline(0.0, linewidth=1.5, linestyle="--")

    # Mark certificate points inside the displayed range.
    cps = np.asarray(result["certificate_points"], dtype=float)
    cps = cps[cps <= y_max]
    if len(cps):
        cp_profits = np.array([portfolio_terminal_profit(y, result) for y in cps])
        ax.scatter(cps, cp_profits, zorder=3, label="finite certificate points")

    ax.fill_between(y_vals, profits, 0.0, where=(profits > 0.0), alpha=0.25, label="positive")
    ax.fill_between(y_vals, profits, 0.0, where=(profits < 0.0), alpha=0.25, label="negative")

    ax.set_xlabel("Terminal stock price S_T")
    ax.set_ylabel("Terminal financed profit Pi(S_T)")
    ax.set_title("Deterministic arbitrage payoff certificate")
    ax.grid(True)
    ax.legend()
    plt.show()


def display_used_quotes(calls: pd.DataFrame, puts: pd.DataFrame, max_rows: int = 50) -> None:
    """Display the effective option quotes passed to the LP."""
    cols = ["type", "strike", "raw_bid", "raw_ask", "eff_bid", "eff_ask"]
    frames = []
    for df, side in [(calls, "call"), (puts, "put")]:
        tmp = _ensure_effective_quote_columns(df, side)
        frames.append(tmp[[c for c in cols if c in tmp.columns]])
    out = pd.concat(frames, ignore_index=True)
    print(f"Displaying first {min(max_rows, len(out))} of {len(out)} quote rows.")
    display(out.head(max_rows))


# =============================================================================
# End-to-end analysis routines
# =============================================================================

def run_analysis_on_quotes(
    spot: float,
    stock_bid: float,
    stock_ask: float,
    calls: pd.DataFrame,
    puts: pd.DataFrame,
    config: UserConfig,
    T: float,
    title: str = "Input summary",
) -> Dict[str, object]:
    """Run the paper LP on already supplied call/put quote tables."""
    calls = _ensure_effective_quote_columns(calls, "call")
    puts = _ensure_effective_quote_columns(puts, "put")

    print_quote_summary(spot, stock_bid, stock_ask, calls, puts, config, T, title=title)

    result = build_and_solve_paper_arbitrage_lp(
        spot=spot,
        stock_bid=stock_bid,
        stock_ask=stock_ask,
        calls=calls,
        puts=puts,
        config=config,
        T=T,
    )

    result["calls"] = calls
    result["puts"] = puts

    print_portfolio_tables(result)
    print_margin_table(result)
    print_conclusion(result)
    plot_profit_function(result)

    return result


def build_config_from_external_inputs(
    ticker: str,
    expiration_date: str,
    transaction_cost_perturbation_percent: float,
    **kwargs,
) -> UserConfig:
    """Build a UserConfig object from editable Colab variables."""
    return UserConfig(
        ticker=ticker.upper().strip(),
        expiration_date=str(expiration_date),
        transaction_cost_perturbation_percent=float(transaction_cost_perturbation_percent),
        **kwargs,
    )


def run_analysis(config: UserConfig, known_expirations: Optional[Sequence[str]] = None) -> Dict[str, object]:
    """Download Yahoo quotes, apply perturbation, and run the paper LP."""
    validate_expiration_date(config.ticker, config.expiration_date, known_expirations)

    spot, calls_raw, puts_raw = fetch_option_chain_from_yahoo(
        ticker=config.ticker,
        expiration_date=config.expiration_date,
        use_last_price=config.use_last_price,
        known_expirations=known_expirations,
    )

    T = year_fraction_to_expiration(config.expiration_date)

    calls = apply_transaction_cost_perturbation(
        calls_raw,
        perturbation_percent=config.transaction_cost_perturbation_percent,
        absolute_perturbation=config.absolute_perturbation,
    )
    puts = apply_transaction_cost_perturbation(
        puts_raw,
        perturbation_percent=config.transaction_cost_perturbation_percent,
        absolute_perturbation=config.absolute_perturbation,
    )

    stock_bid, stock_ask = effective_stock_bid_ask(
        spot=spot,
        stock_spread_percent=config.stock_spread_percent,
        perturbation_percent=config.transaction_cost_perturbation_percent,
    )

    return run_analysis_on_quotes(
        spot=spot,
        stock_bid=stock_bid,
        stock_ask=stock_ask,
        calls=calls,
        puts=puts,
        config=config,
        T=T,
        title="Yahoo input summary",
    )


# =============================================================================
# Interactive Colab workflow
# =============================================================================

def _read_nonempty_text(prompt: str) -> str:
    """Read a non-empty string from the user."""
    while True:
        value = input(prompt).strip()
        if value:
            return value
        print("Please enter a non-empty value.")


def _read_float(prompt: str, default: Optional[float] = None, minimum: Optional[float] = None) -> float:
    """Read a floating-point value, with optional default and lower bound."""
    while True:
        suffix = f" [default: {default}]" if default is not None else ""
        raw = input(prompt + suffix + ": ").strip()
        if raw == "" and default is not None:
            value = float(default)
        else:
            try:
                value = float(raw)
            except ValueError:
                print("Please enter a valid number.")
                continue
        if minimum is not None and value < minimum:
            print(f"Please enter a value >= {minimum}.")
            continue
        return value


def _read_bool(prompt: str, default: bool = False) -> bool:
    """Read a yes/no answer from the user."""
    default_text = "y" if default else "n"
    while True:
        raw = input(f"{prompt} [y/n, default: {default_text}]: ").strip().lower()
        if raw == "":
            return default
        if raw in {"y", "yes"}:
            return True
        if raw in {"n", "no"}:
            return False
        print("Please answer y or n.")


def _choose_expiration_date_interactively(ticker: str, dates: Sequence[str]) -> str:
    """Ask the user to choose an expiration date by index or exact date."""
    if not dates:
        raise ValueError("No expiration dates are available.")

    while True:
        raw = input("Choose expiration date by number or exact YYYY-MM-DD date: ").strip()
        if raw.isdigit():
            idx = int(raw)
            if 0 <= idx < len(dates):
                return str(dates[idx])
            print(f"Please enter a number between 0 and {len(dates) - 1}.")
            continue
        if raw in dates:
            return raw
        print(f"{raw!r} is not one of the available expiration dates for {ticker.upper()}.")


def run_interactive_analysis(
    default_risk_free_rate_percent: float = 4.0,
    default_setup_cost_budget: float = 0.0,
    default_use_last_price: bool = False,
    default_stock_spread_percent: float = 0.0,
    default_absolute_perturbation: float = 0.0,
    ask_advanced_settings: bool = False,
) -> Dict[str, object]:
    """Run the full Yahoo workflow through Colab prompts."""
    print("=" * 78)
    print("Interactive deterministic arbitrage search")
    print("=" * 78)
    print("This workflow solves the paper LP with bid-ask execution.")
    print("For the exact deterministic-arbitrage test, keep setup_cost_budget = 0.\n")

    ticker = _read_nonempty_text("Enter ticker symbol, e.g. AAPL: ").upper().strip()

    print("\nFetching available expiration dates...")
    dates = show_available_expiration_dates(ticker)
    expiration_date = _choose_expiration_date_interactively(ticker, dates)

    perturbation = _read_float(
        "Enter transaction-cost perturbation percent. Example: 1 means asks increase by 1% and bids decrease by 1%",
        default=0.0,
        minimum=0.0,
    )

    if ask_advanced_settings:
        risk_free_rate_percent = _read_float(
            "Annual continuously compounded risk-free rate in percent",
            default=default_risk_free_rate_percent,
        )
        setup_cost_budget = _read_float(
            "Setup-cost budget Y. Use 0 for the paper arbitrage test",
            default=default_setup_cost_budget,
        )
        use_last_price = _read_bool(
            "Use lastPrice as both bid and ask before perturbation?",
            default=default_use_last_price,
        )
        stock_spread_percent = _read_float(
            "Initial stock spread percent around spot",
            default=default_stock_spread_percent,
            minimum=0.0,
        )
        absolute_perturbation = _read_float(
            "Absolute perturbation in price units",
            default=default_absolute_perturbation,
            minimum=0.0,
        )
    else:
        risk_free_rate_percent = float(default_risk_free_rate_percent)
        setup_cost_budget = float(default_setup_cost_budget)
        use_last_price = bool(default_use_last_price)
        stock_spread_percent = float(default_stock_spread_percent)
        absolute_perturbation = float(default_absolute_perturbation)

    cfg = build_config_from_external_inputs(
        ticker=ticker,
        expiration_date=expiration_date,
        transaction_cost_perturbation_percent=perturbation,
        use_last_price=use_last_price,
        risk_free_rate_percent=risk_free_rate_percent,
        setup_cost_budget=setup_cost_budget,
        absolute_perturbation=absolute_perturbation,
        stock_spread_percent=stock_spread_percent,
    )

    print("\nSelected inputs")
    print("-" * 78)
    print(f"Ticker: {cfg.ticker}")
    print(f"Expiration date: {cfg.expiration_date}")
    print(f"Transaction-cost perturbation: {cfg.transaction_cost_perturbation_percent}%")
    print(f"Risk-free rate: {cfg.risk_free_rate_percent}%")
    print(f"Setup-cost budget: {cfg.setup_cost_budget}")
    print("-" * 78)

    return run_analysis(cfg, known_expirations=dates)


# =============================================================================
# Paper Section 7.5 bid-ask put-call inconsistency example
# =============================================================================

def make_paper_three_strike_bid_ask_example() -> Tuple[float, float, float, pd.DataFrame, pd.DataFrame, UserConfig, float]:
    """Create the three-strike bid-ask example from the paper.

    The quotes contain an executable put-call inconsistency at K=100:
        C_bid(100) - P_ask(100) = 8.80 - 8.10 = 0.70 > S0 - K = 0.

    The arbitrage described in the paper is:
        sell call K=100 at bid,
        buy put K=100 at ask,
        buy one share at ask/spot,
        borrow the balancing cash amount.
    """
    S0 = 100.0
    stock_bid = 100.0
    stock_ask = 100.0
    T = 1.0

    calls = pd.DataFrame({
        "type": ["call", "call", "call"],
        "strike": [90.0, 100.0, 110.0],
        "raw_bid": [13.80, 8.80, 4.80],
        "raw_ask": [14.20, 9.20, 5.20],
    })
    puts = pd.DataFrame({
        "type": ["put", "put", "put"],
        "strike": [90.0, 100.0, 110.0],
        "raw_bid": [3.70, 7.70, 14.80],
        "raw_ask": [4.10, 8.10, 15.20],
    })
    calls["eff_bid"] = calls["raw_bid"]
    calls["eff_ask"] = calls["raw_ask"]
    puts["eff_bid"] = puts["raw_bid"]
    puts["eff_ask"] = puts["raw_ask"]

    cfg = UserConfig(
        ticker="PAPER_EXAMPLE",
        expiration_date="T=1",
        transaction_cost_perturbation_percent=0.0,
        risk_free_rate_percent=0.0,
        setup_cost_budget=0.0,
        max_total_call_options_to_buy=3.0,
        max_total_call_options_to_sell=3.0,
        max_total_put_options_to_buy=3.0,
        max_total_put_options_to_sell=3.0,
        max_call_options_to_buy_per_contract=1.0,
        max_call_options_to_sell_per_contract=1.0,
        max_put_options_to_buy_per_contract=1.0,
        max_put_options_to_sell_per_contract=1.0,
        max_shares_to_buy=1.0,
        max_shares_to_sell=1.0,
        max_deposit_limit=1_000.0,
        max_borrow_limit=1_000.0,
        eta_tail_weight=1.0,
        high_price_multiplier=2.0,
    )

    return S0, stock_bid, stock_ask, calls, puts, cfg, T


def run_paper_three_strike_demo() -> Dict[str, object]:
    """Run the paper's three-strike bid-ask example."""
    S0, stock_bid, stock_ask, calls, puts, cfg, T = make_paper_three_strike_bid_ask_example()
    return run_analysis_on_quotes(
        spot=S0,
        stock_bid=stock_bid,
        stock_ask=stock_ask,
        calls=calls,
        puts=puts,
        config=cfg,
        T=T,
        title="Paper three-strike bid-ask example",
    )


def print_explicit_paper_put_call_arbitrage() -> pd.DataFrame:
    """Print the explicit put-call arbitrage described in the paper.

    The executable negative-cost portfolio in the paper is:
        sell 1 call K=100 at bid 8.80,
        buy 1 put K=100 at ask 8.10,
        buy 1 share at 100,
        borrow 100 at r=0.

    Its setup cost is -0.70 and its terminal option-stock-bank payoff is 0.
    Investing the immediate inflow 0.70 converts it into a zero-cost portfolio
    with terminal payoff 0.70 in every state.

    The zero-cost version shown below borrows only 99.30 instead of 100.
    Its setup cost is exactly zero and its payoff is exactly 0.70 for all x.
    """
    K = 100.0
    call_bid = 8.80
    put_ask = 8.10
    stock_ask = 100.0
    bank = -99.30

    setup_cost = -call_bid + put_ask + stock_ask + bank
    states = np.array([0.0, 90.0, 100.0, 110.0, 150.0])
    payoff = -np.maximum(states - K, 0.0) + np.maximum(K - states, 0.0) + states + bank

    df = pd.DataFrame({
        "S_T": states,
        "zero_cost_payoff": payoff,
    })

    print("\n" + "=" * 78)
    print("Explicit paper put-call arbitrage at K=100")
    print("=" * 78)
    print("Trade: sell call bid 8.80, buy put ask 8.10, buy stock 100, bank = -99.30.")
    print(f"Executable setup cost: {setup_cost:.10f}")
    print("Terminal payoff is 0.70 at every displayed state and, by put-call identity, at every state.")
    display(df)
    return df


# =============================================================================
# Localized target-positive arbitrage construction
# =============================================================================

# The paper also introduces localized arbitrage portfolios positive on a target
# set A. The helper below implements the common lower-margin variant described
# in Remark 5.3: maximize a single t subject to Pi(y) >= t at all certificate
# points y in E(A). If t > 0, the returned globally nonnegative portfolio is
# strictly positive throughout the selected interval union.


def _parse_finite_left_endpoint(raw_text, interval_number):
    """Parse a finite left endpoint a_j."""
    text = str(raw_text).strip().lower().replace(",", ".")
    try:
        value = float(text)
    except ValueError as exc:
        raise ValueError(f"Interval {interval_number}: left endpoint must be finite.") from exc
    if not np.isfinite(value) or value < 0:
        raise ValueError(f"Interval {interval_number}: left endpoint must be finite and >= 0.")
    return float(value)


def _parse_right_endpoint(raw_text, interval_number):
    """Parse a finite or infinite right endpoint b_j."""
    text = str(raw_text).strip().lower().replace(",", ".")
    infinite_tokens = {"inf", "+inf", "infinity", "+infinity", "infty", "+infty", "oo", "+oo"}
    if text in infinite_tokens:
        return float("inf")
    try:
        value = float(text)
    except ValueError as exc:
        raise ValueError(f"Interval {interval_number}: right endpoint must be a number or inf.") from exc
    if value < 0:
        raise ValueError(f"Interval {interval_number}: right endpoint must be >= 0.")
    return float(value)


def _read_positive_integer(prompt):
    """Read a positive integer from input()."""
    raw = input(prompt).strip()
    try:
        value = int(raw)
    except ValueError as exc:
        raise ValueError("The number of intervals must be a positive integer.") from exc
    if value <= 0:
        raise ValueError("The number of intervals must be at least 1.")
    return value


def _read_intervals_interactively():
    """Ask the user for a finite union of open intervals."""
    q = _read_positive_integer("How many intervals do you want to use? ")
    intervals = []
    for j in range(1, q + 1):
        print(f"\nInterval A_{j}.")
        print("Use a finite right endpoint, e.g. 150, or type inf for +infty.")
        a = _parse_finite_left_endpoint(input(f"Enter left endpoint a_{j}: "), j)
        b = _parse_right_endpoint(input(f"Enter right endpoint b_{j}: "), j)
        if np.isfinite(b) and not a < b:
            raise ValueError(f"Interval {j}: a finite interval must satisfy a < b.")
        intervals.append({"index": j, "a": float(a), "b": float(b), "unbounded": bool(np.isinf(b))})
    return intervals


def _format_interval(interval):
    """Return a readable label for one target interval."""
    a = float(interval["a"])
    if interval["unbounded"]:
        return f"A_{interval['index']} = ({a:.6f}, +infty)"
    return f"A_{interval['index']} = ({a:.6f}, {float(interval['b']):.6f})"


def _make_target_certificate_points(intervals, all_strikes):
    """Build E(A): endpoints plus strikes inside selected intervals."""
    point_sources = {}

    def add_point(value, source):
        key = round(float(value), 12)
        if key not in point_sources:
            point_sources[key] = {"value": float(value), "sources": []}
        point_sources[key]["sources"].append(source)

    for interval in intervals:
        j = int(interval["index"])
        a = float(interval["a"])
        b = float(interval["b"])
        add_point(a, f"A_{j} left endpoint")
        if interval["unbounded"]:
            strikes_inside = all_strikes[all_strikes > a]
        else:
            add_point(b, f"A_{j} right endpoint")
            strikes_inside = all_strikes[(all_strikes > a) & (all_strikes < b)]
        for strike in strikes_inside:
            add_point(float(strike), f"A_{j} strike")

    items = sorted(point_sources.values(), key=lambda item: item["value"])
    points = np.array([item["value"] for item in items], dtype=float)
    labels = [f"x_{k+1}: " + "; ".join(item["sources"]) for k, item in enumerate(items)]
    return points, labels


def build_and_solve_target_positive_arbitrage(
    base_result: Dict[str, object],
    intervals: Sequence[Dict[str, object]],
) -> Dict[str, object]:
    """Construct a globally nonnegative portfolio positive on target intervals.

    This solves the common-margin version of the paper's localized LP:
        maximize t
    subject to
        Pi(x) >= 0 globally,
        Pi(y) >= t for every target certificate point y in E(A),
        t >= 0.

    If t > 0, the payoff is positive on the selected interval union.
    """
    if base_result is None:
        raise RuntimeError("Run the main arbitrage analysis before the target-positive solver.")
    if not intervals:
        raise ValueError("At least one interval is required.")

    config = base_result["config"]
    call_strikes = np.asarray(base_result["call_strikes"], dtype=float)
    put_strikes = np.asarray(base_result["put_strikes"], dtype=float)
    all_strikes = np.asarray(base_result["all_strikes"], dtype=float)
    certificate_points = np.asarray(base_result["certificate_points"], dtype=float)
    target_points, target_labels = _make_target_certificate_points(intervals, all_strikes)
    if len(target_points) == 0:
        raise RuntimeError("No target certificate points were generated.")

    m = len(call_strikes)
    n = len(put_strikes)
    q_global = len(certificate_points)
    base_layout = base_result["layout"]
    n_portfolio = base_layout["n_portfolio"]

    # Variables are the same portfolio variables plus one common target margin t.
    t_idx = n_portfolio
    L = n_portfolio + 1

    setup_budget = float(base_result.get("setup_budget", 0.0))
    bank_growth = float(base_result["bank_growth"])

    c = np.zeros(L, dtype=float)
    c[t_idx] = -1.0  # maximize t by minimizing -t

    # Cost equality.
    cost_coeff = _initial_cost_coefficients(
        call_ask=base_result["call_ask"],
        call_bid=base_result["call_bid"],
        put_ask=base_result["put_ask"],
        put_bid=base_result["put_bid"],
        stock_ask=base_result["stock_ask"],
        stock_bid=base_result["stock_bid"],
    )
    A_eq = np.zeros((1, L), dtype=float)
    A_eq[0, :n_portfolio] = cost_coeff
    b_eq = np.array([setup_budget], dtype=float)

    rows = []
    rhs = []

    # Global nonnegativity at 0 and all strikes: Pi(y) >= 0.
    for y in certificate_points:
        row = np.zeros(L, dtype=float)
        row[:n_portfolio] = -_portfolio_payoff_coefficients(
            y,
            call_strikes,
            put_strikes,
            bank_growth,
        )
        rows.append(row)
        rhs.append(-setup_budget * bank_growth)

    # Target positivity: Pi(y) >= t.
    for y in target_points:
        row = np.zeros(L, dtype=float)
        row[:n_portfolio] = -_portfolio_payoff_coefficients(
            y,
            call_strikes,
            put_strikes,
            bank_growth,
        )
        row[t_idx] = 1.0
        rows.append(row)
        rhs.append(-setup_budget * bank_growth)

    # Tail slope must be nonnegative for global nonnegativity.
    slope_coeff = _tail_slope_coefficients(m, n)
    row = np.zeros(L, dtype=float)
    row[:n_portfolio] = -slope_coeff
    rows.append(row)
    rhs.append(0.0)

    # Same total position limits as in the main LP.
    row = np.zeros(L, dtype=float); row[0:m] = 1.0
    rows.append(row); rhs.append(float(config.max_total_call_options_to_buy))
    row = np.zeros(L, dtype=float); row[m:2*m] = 1.0
    rows.append(row); rhs.append(float(config.max_total_call_options_to_sell))
    row = np.zeros(L, dtype=float); row[2*m:2*m+n] = 1.0
    rows.append(row); rhs.append(float(config.max_total_put_options_to_buy))
    row = np.zeros(L, dtype=float); row[2*m+n:2*m+2*n] = 1.0
    rows.append(row); rhs.append(float(config.max_total_put_options_to_sell))

    A_ub = np.vstack(rows)
    b_ub = np.asarray(rhs, dtype=float)

    bounds: List[Tuple[float, Optional[float]]] = []
    bounds += [(0.0, float(config.max_call_options_to_buy_per_contract)) for _ in range(m)]
    bounds += [(0.0, float(config.max_call_options_to_sell_per_contract)) for _ in range(m)]
    bounds += [(0.0, float(config.max_put_options_to_buy_per_contract)) for _ in range(n)]
    bounds += [(0.0, float(config.max_put_options_to_sell_per_contract)) for _ in range(n)]
    bounds.append((0.0, float(config.max_shares_to_buy)))
    bounds.append((0.0, float(config.max_shares_to_sell)))
    bounds.append((-float(config.max_borrow_limit), float(config.max_deposit_limit)))
    bounds.append((0.0, None))  # t

    res = linprog(
        c=c,
        A_ub=A_ub,
        b_ub=b_ub,
        A_eq=A_eq,
        b_eq=b_eq,
        bounds=bounds,
        method="highs",
    )

    if not res.success or res.x is None:
        raise RuntimeError(
            "The target-positive LP did not return an optimal solution. "
            f"Status={res.status}, message={res.message}"
        )

    x = np.asarray(res.x, dtype=float)
    focused = dict(base_result)
    focused["solution"] = np.concatenate([x[:n_portfolio], np.zeros(q_global + 1)])
    focused["target_t"] = float(x[t_idx])
    focused["target_points"] = target_points
    focused["target_labels"] = target_labels
    focused["intervals"] = list(intervals)
    focused["objective_value"] = float(x[t_idx])
    focused["layout"] = base_layout
    return focused


def run_interactive_target_positive_analysis(base_result: Dict[str, object]) -> Dict[str, object]:
    """Interactive wrapper for the localized target-positive LP."""
    intervals = _read_intervals_interactively()
    focused = build_and_solve_target_positive_arbitrage(base_result, intervals)

    print("\n" + "=" * 78)
    print("Target-positive arbitrage result")
    print("=" * 78)
    print("Selected intervals:")
    for interval in intervals:
        print("  -", _format_interval(interval))
    print(f"Maximized common target margin t*: {focused['target_t']:.10f}")

    rows = []
    for label, y in zip(focused["target_labels"], focused["target_points"]):
        rows.append({
            "Point": label,
            "S_T": float(y),
            "Pi(S_T)": portfolio_terminal_profit(float(y), focused),
        })
    display(pd.DataFrame(rows))

    print_portfolio_tables(focused)
    tol = float(base_result["config"].solver_tolerance)
    if focused["target_t"] > tol:
        print("\n[CONCLUSION] The portfolio is globally nonnegative and positive on the selected intervals.")
    else:
        print("\n[CONCLUSION] No strictly target-positive portfolio was found within the imposed bounds.")

    plot_profit_function(focused)
    return focused



## Cell 3 -- Run the paper's three-strike bid--ask example

This example is included so that the notebook can be tested without downloading market data.

The quotes are the paper's three-strike bid--ask example. At strike 100,

$$
C^{bid}_{100}-P^{ask}_{100}=8.80-8.10=0.70>S_0-K=0,
$$

so there is an executable put--call deterministic arbitrage.


In [ ]:
# Run this cell to verify that the implementation detects the paper's
# three-strike bid-ask put-call arbitrage.
#
# First, print the explicit put-call trade from the paper. Then solve the
# general paper LP. The optimizer may select a different valid arbitrage if
# it has larger breakpoint/tail margins under the imposed bounds.
explicit_paper_trade = print_explicit_paper_put_call_arbitrage()
demo_results = run_paper_three_strike_demo()



## Cell 4 -- Run the interactive Yahoo Finance arbitrage analysis

Run this cell when you want to test a real option chain from Yahoo Finance.

The workflow asks for:

1. ticker symbol;
2. expiration date, selected from Yahoo's available dates;
3. transaction-cost perturbation percentage.

By default the exact paper test is used: `setup_cost_budget = 0.0`, continuous positions, continuous compounding, and bid--ask execution.


In [ ]:

# Standard interactive run.
# This cell will ask for ticker, expiration date, and perturbation percentage.
results = run_interactive_analysis()


## Cell 5 -- Construct an arbitrage portfolio on selected intervals

Run this cell immediately after Cell 4 has created the variable `results`.

This cell restores the localized interval workflow from the previous notebook, but uses the paper-consistent LP and executable bid--ask convention. The default `paper_sum_D` mode reproduces the old/direct Section-5 objective with one pointwise margin `D_y` per target certificate point. For the stricter certificate that forces one common lower margin on all selected points, set `LOCALIZED_LP_MODE = "common_t"` in the code cell.


In [ ]:
# =============================================================================
# Cell 5 - Construct a localized arbitrage portfolio on selected intervals
# =============================================================================
#
# Run this cell after the main analysis cell:
#
#     results = run_interactive_analysis()
#
# The variable "results" contains the quotes, strikes, bid-ask prices, risk-free
# rate, position bounds, and helper data produced by the paper-consistent LP.
#
# Mathematical target
# -------------------
# You choose a finite union of intervals
#
#     A = union_j A_j,
#
# where each A_j is either a finite interval (a_j, b_j) or a right-unbounded
# interval (a_j, +infty). This cell searches for a zero-setup-cost portfolio
# whose terminal financed profit Pi(S_T) satisfies:
#
#     Pi(x) >= 0 for every x >= 0,
#     Pi(x) >  0 on the selected interval union A, if the certificate succeeds.
#
# This is the localized version of the deterministic arbitrage construction in
# Section 5 of the paper. The LP remains finite because every payoff is
# continuous and piecewise affine in x = S_T, with breakpoints only at traded
# strikes.
#
# Bid-ask convention
# ------------------
# The portfolio variables are the same executable variables as in the main LP:
#
#     call_buy_i  >= 0, paid at call ask_i;
#     call_sell_i >= 0, received at call bid_i;
#     put_buy_i   >= 0, paid at put ask_i;
#     put_sell_i  >= 0, received at put bid_i;
#     share_buy   >= 0, paid at stock ask;
#     share_sell  >= 0, received at stock bid;
#     bank is a signed cash-account position.
#
# Hence the returned trade is executable: long positions are costed at ask and
# short positions are credited at bid.
#
# Certificate points for selected intervals
# -----------------------------------------
# For a finite interval (a,b), the target certificate set contains:
#
#     a, b, and every traded strike K such that a < K < b.
#
# For a right-unbounded interval (a,+infty), it contains:
#
#     a and every traded strike K such that K > a.
#
# There is no artificial point at infinity. The tail is controlled by the same
# nonnegative right-tail slope condition used by the global detector:
#
#     net shares + sum(net calls) >= 0.
#
# Available localized objectives
# ------------------------------
# 1. common_t     (default and recommended for interval positivity)
#
#        maximize t
#        subject to Pi(y) >= t for every y in E(A), t >= 0.
#
#    If t > 0, this gives a clean certificate that Pi is positive on all
#    selected intervals.
#
# 2. paper_sum_D  (old-cell / direct Section-5 style)
#
#        maximize sum_y w_y D_y
#        subject to Pi(y) >= D_y, D_y >= 0.
#
#    This matches the older notebook's logic. If every D_y > 0, interval
#    positivity is certified. If some D_y is zero, the portfolio is still
#    globally nonnegative, but full positivity on the whole interval union is
#    not certified.
#
# Both modes use scipy.optimize.linprog and continuous quantities, as in the
# paper. No integer lot constraints are imposed here.


def build_and_solve_localized_interval_arbitrage(
    base_result,
    intervals,
    mode="paper_sum_D",
    target_weights=None,
):
    """Solve a localized interval-arbitrage LP.

    Parameters
    ----------
    base_result:
        Dictionary returned by run_interactive_analysis(), run_analysis_on_quotes(),
        or run_paper_three_strike_demo().

    intervals:
        List of dictionaries with keys index, a, b, unbounded. The helper
        _read_intervals_interactively(), defined in the implementation cell,
        returns this exact format.

    mode:
        "common_t" or "paper_sum_D".

    target_weights:
        Optional positive weights for paper_sum_D mode. If omitted, all weights
        are one, matching the old notebook.

    Returns
    -------
    focused_result:
        A result dictionary compatible with the existing payoff, portfolio-table,
        and plotting helpers.
    """
    if base_result is None:
        raise RuntimeError(
            "Run the main arbitrage analysis first, e.g. results = run_interactive_analysis()."
        )
    if not intervals:
        raise ValueError("At least one target interval is required.")

    mode_key = str(mode).strip().lower()
    if mode_key not in {"common_t", "paper_sum_d"}:
        raise ValueError("mode must be either 'common_t' or 'paper_sum_D'.")

    config = base_result["config"]

    # Already-cleaned quote arrays from the main result. These are the effective
    # executable quotes after Yahoo cleaning and any optional artificial spread
    # perturbation.
    call_strikes = np.asarray(base_result["call_strikes"], dtype=float)
    put_strikes = np.asarray(base_result["put_strikes"], dtype=float)
    all_strikes = np.asarray(base_result["all_strikes"], dtype=float)
    certificate_points = np.asarray(base_result["certificate_points"], dtype=float)

    m = len(call_strikes)
    n = len(put_strikes)

    # Build E(A): interval endpoints plus traded strikes inside the selected
    # intervals. This helper is defined in the implementation cell and follows
    # the Section-5 finite certificate construction.
    target_points, target_labels = _make_target_certificate_points(intervals, all_strikes)
    n_targets = len(target_points)
    if n_targets == 0:
        raise RuntimeError("No target certificate points were generated.")

    # Weights are used only in paper_sum_D mode. Keeping them available makes
    # the old behavior extensible: one may prioritize some intervals/points.
    if target_weights is None:
        weights = np.ones(n_targets, dtype=float)
    else:
        weights = np.asarray(target_weights, dtype=float)
        if weights.shape != (n_targets,):
            raise ValueError(
                f"target_weights must have length {n_targets}, not shape {weights.shape}."
            )
        if np.any(weights <= 0.0) or not np.all(np.isfinite(weights)):
            raise ValueError("All target weights must be finite and strictly positive.")

    base_layout = base_result["layout"]
    n_portfolio = int(base_layout["n_portfolio"])

    # LP variable layout for this localized problem.
    #
    #   x[0:n_portfolio] are executable portfolio variables.
    #   In paper_sum_D mode, x[n_portfolio:] are D_y variables.
    #   In common_t mode, x[n_portfolio] is the scalar common margin t.
    if mode_key == "paper_sum_d":
        margin_start = n_portfolio
        L = n_portfolio + n_targets
    else:
        t_idx = n_portfolio
        L = n_portfolio + 1

    setup_budget = float(base_result.get("setup_budget", 0.0))
    bank_growth = float(base_result["bank_growth"])

    # -------------------------------------------------------------------------
    # Objective.
    # -------------------------------------------------------------------------
    # scipy.optimize.linprog minimizes, so every maximization objective is
    # multiplied by -1.
    c = np.zeros(L, dtype=float)
    if mode_key == "paper_sum_d":
        c[margin_start:margin_start+n_targets] = -weights
    else:
        c[t_idx] = -1.0

    # -------------------------------------------------------------------------
    # Equality: executable setup cost equals the chosen setup budget.
    # -------------------------------------------------------------------------
    cost_coeff = _initial_cost_coefficients(
        call_ask=base_result["call_ask"],
        call_bid=base_result["call_bid"],
        put_ask=base_result["put_ask"],
        put_bid=base_result["put_bid"],
        stock_ask=base_result["stock_ask"],
        stock_bid=base_result["stock_bid"],
    )
    A_eq = np.zeros((1, L), dtype=float)
    A_eq[0, :n_portfolio] = cost_coeff
    b_eq = np.array([setup_budget], dtype=float)

    # -------------------------------------------------------------------------
    # Inequalities A_ub x <= b_ub.
    # -------------------------------------------------------------------------
    rows = []
    rhs = []

    # 1) Global nonnegativity at the finite global certificate points:
    #       Pi(0) >= 0 and Pi(K) >= 0 for every traded strike K.
    # In A_ub form:
    #       -G(y) <= -setup_budget * exp(rT).
    for y in certificate_points:
        row = np.zeros(L, dtype=float)
        row[:n_portfolio] = -_portfolio_payoff_coefficients(
            float(y),
            call_strikes,
            put_strikes,
            bank_growth,
        )
        rows.append(row)
        rhs.append(-setup_budget * bank_growth)

    # 2) Target interval constraints.
    #
    # paper_sum_D mode:
    #       Pi(y_j) >= D_j.
    #
    # common_t mode:
    #       Pi(y_j) >= t.
    for j, y in enumerate(target_points):
        row = np.zeros(L, dtype=float)
        row[:n_portfolio] = -_portfolio_payoff_coefficients(
            float(y),
            call_strikes,
            put_strikes,
            bank_growth,
        )
        if mode_key == "paper_sum_d":
            row[margin_start + j] = 1.0
        else:
            row[t_idx] = 1.0
        rows.append(row)
        rhs.append(-setup_budget * bank_growth)

    # 3) Right-tail slope guard:
    #       net shares + sum(net calls) >= 0.
    slope_coeff = _tail_slope_coefficients(m, n)
    row = np.zeros(L, dtype=float)
    row[:n_portfolio] = -slope_coeff
    rows.append(row)
    rhs.append(0.0)

    # 4) Same total quantity bounds as the main paper LP.
    # These are normalization/liquidity bounds, not no-arbitrage assumptions.
    row = np.zeros(L, dtype=float)
    row[0:m] = 1.0
    rows.append(row)
    rhs.append(float(config.max_total_call_options_to_buy))

    row = np.zeros(L, dtype=float)
    row[m:2*m] = 1.0
    rows.append(row)
    rhs.append(float(config.max_total_call_options_to_sell))

    row = np.zeros(L, dtype=float)
    row[2*m:2*m+n] = 1.0
    rows.append(row)
    rhs.append(float(config.max_total_put_options_to_buy))

    row = np.zeros(L, dtype=float)
    row[2*m+n:2*m+2*n] = 1.0
    rows.append(row)
    rhs.append(float(config.max_total_put_options_to_sell))

    A_ub = np.vstack(rows)
    b_ub = np.asarray(rhs, dtype=float)

    # -------------------------------------------------------------------------
    # Variable bounds.
    # -------------------------------------------------------------------------
    bounds = []

    # Option buy/sell variables, bounded per contract.
    bounds += [(0.0, float(config.max_call_options_to_buy_per_contract)) for _ in range(m)]
    bounds += [(0.0, float(config.max_call_options_to_sell_per_contract)) for _ in range(m)]
    bounds += [(0.0, float(config.max_put_options_to_buy_per_contract)) for _ in range(n)]
    bounds += [(0.0, float(config.max_put_options_to_sell_per_contract)) for _ in range(n)]

    # Stock buy/sell variables.
    bounds.append((0.0, float(config.max_shares_to_buy)))
    bounds.append((0.0, float(config.max_shares_to_sell)))

    # Bank account. Positive means deposit; negative means borrow.
    bounds.append((-float(config.max_borrow_limit), float(config.max_deposit_limit)))

    # Localized margin variables: D_y >= 0 or t >= 0.
    if mode_key == "paper_sum_d":
        bounds += [(0.0, None) for _ in range(n_targets)]
    else:
        bounds.append((0.0, None))

    res = linprog(
        c=c,
        A_ub=A_ub,
        b_ub=b_ub,
        A_eq=A_eq,
        b_eq=b_eq,
        bounds=bounds,
        method="highs",
    )

    if not res.success or res.x is None:
        raise RuntimeError(
            "The localized interval LP did not return an optimal solution. "
            f"Status={res.status}, message={res.message}"
        )

    x = np.asarray(res.x, dtype=float)
    portfolio_solution = x[:n_portfolio]

    if mode_key == "paper_sum_d":
        target_D_values = x[margin_start:margin_start+n_targets]
        target_t = float(np.min(target_D_values)) if len(target_D_values) else 0.0
        objective_label = "sum_y w_y D_y"
        objective_value = float(-res.fun)
    else:
        target_t = float(x[t_idx])
        target_D_values = np.full(n_targets, target_t, dtype=float)
        objective_label = "common target margin t"
        objective_value = target_t

    # The generic helpers expect result["solution"] to have the same layout as
    # the main detector: portfolio variables followed by the main detector's
    # breakpoint margins and tail margin. The localized LP has different margin
    # variables, so store them separately and pad the main solution with zeros.
    q_global = len(certificate_points)
    compatible_solution = np.concatenate([
        portfolio_solution,
        np.zeros(q_global + 1, dtype=float),
    ])

    focused = dict(base_result)
    focused["solution"] = compatible_solution
    focused["localized_lp_status"] = res.status
    focused["localized_lp_message"] = res.message
    focused["localized_mode"] = mode_key
    focused["localized_objective_label"] = objective_label
    focused["localized_objective_value"] = objective_value
    focused["localized_target_weights"] = weights
    focused["target_D_values"] = target_D_values
    focused["target_t"] = target_t
    focused["target_points"] = target_points
    focused["target_labels"] = target_labels
    focused["intervals"] = list(intervals)
    focused["layout"] = base_layout
    focused["objective_value"] = objective_value

    # Expand plotting range if selected intervals are far from the default range.
    finite_right_endpoints = [
        float(interval["b"])
        for interval in intervals
        if not bool(interval.get("unbounded", False))
    ]
    max_interval_endpoint = max(finite_right_endpoints) if finite_right_endpoints else 0.0
    max_target = float(np.max(target_points)) if len(target_points) else 0.0
    focused["S_high"] = max(
        float(base_result.get("S_high", 0.0)),
        float(base_result.get("spot", 0.0)) * 1.5,
        max_interval_endpoint * 1.25,
        max_target * 1.25,
        1.0,
    )

    return focused



def localized_interval_certificate_table(focused_result):
    """Return the target certificate table as a pandas DataFrame."""
    rows = []
    for label, y, d in zip(
        focused_result["target_labels"],
        focused_result["target_points"],
        focused_result["target_D_values"],
    ):
        pi_y = portfolio_terminal_profit(float(y), focused_result)
        rows.append({
            "Point": label,
            "S_T": float(y),
            "Certified margin": float(d),
            "Pi(S_T)": float(pi_y),
            "Pi - margin": float(pi_y - d),
        })
    return pd.DataFrame(rows)



def print_localized_interval_arbitrage_report(focused_result):
    """Print the localized LP result, target margins, and portfolio tables."""
    config = focused_result["config"]
    tol = float(config.solver_tolerance)
    D_values = np.asarray(focused_result["target_D_values"], dtype=float)
    setup_budget = float(focused_result.get("setup_budget", 0.0))

    print("\n" + "=" * 78)
    print("Localized interval arbitrage result")
    print("=" * 78)
    print("Solver status:", focused_result["localized_lp_status"])
    print("Solver message:", focused_result["localized_lp_message"])
    print("Mode:", focused_result["localized_mode"])
    print(f"Objective ({focused_result['localized_objective_label']}):", round(float(focused_result["localized_objective_value"]), 10))

    print("\nSelected intervals:")
    for interval in focused_result["intervals"]:
        print("  -", _format_interval(interval))

    print("\nNumber of target certificate points:", len(focused_result["target_points"]))
    if len(D_values):
        print("Minimum certified target margin:", round(float(np.min(D_values)), 10))
        print("Maximum certified target margin:", round(float(np.max(D_values)), 10))
    print("Executable setup cost:", round(executable_setup_cost(focused_result), 10))
    print("Right-tail slope:", round(portfolio_tail_slope(focused_result), 10))

    print("\n" + "=" * 78)
    print("Target certificate margins")
    print("=" * 78)
    certificate_df = localized_interval_certificate_table(focused_result)
    display(certificate_df)

    print_portfolio_tables(focused_result)

    print("\n" + "=" * 78)
    print("Conclusion for the selected intervals")
    print("=" * 78)

    if abs(setup_budget) > tol:
        print("[NOTE] setup_cost_budget is not zero, so this is a financed-payoff certificate.")
        print("For the pure deterministic-arbitrage test, use setup_cost_budget = 0.0.")

    if len(D_values) == 0 or float(focused_result["localized_objective_value"]) <= tol:
        print("[CONCLUSION] No positive payoff was found at the selected target certificate points.")
        print("Within the imposed bounds, the localized LP returned only zero target margins.")
    elif np.all(D_values > tol) and abs(setup_budget) <= tol:
        print("[CONCLUSION] Localized deterministic arbitrage certified.")
        print("The portfolio has zero executable setup cost, Pi(x) >= 0 for every S_T >= 0,")
        print("and Pi(x) > 0 throughout every selected interval.")
    elif np.all(D_values > tol):
        print("[CONCLUSION] Localized positivity was certified for the financed payoff Pi(x).")
        print("Set setup_cost_budget = 0.0 for the pure zero-cost deterministic arbitrage test.")
    else:
        print("[CONCLUSION] The portfolio is globally nonnegative and has positive target objective,")
        print("but strict positivity was not certified at every target certificate point.")
        print("Near-zero target margins:")
        for label, y, d in zip(
            focused_result["target_labels"],
            focused_result["target_points"],
            D_values,
        ):
            if float(d) <= tol:
                print(f"  - {label} at S_T={float(y):.8f}, margin={float(d):.10f}")


def plot_localized_interval_profit_function(focused_result):
    """Plot Pi(x) and shade the selected intervals used by the localized LP."""
    config = focused_result["config"]
    intervals = focused_result["intervals"]
    y_max = max(float(focused_result["S_high"]), 1.0)
    y_vals = np.linspace(0.0, y_max, int(config.plot_points))
    profits = np.array([portfolio_terminal_profit(float(y), focused_result) for y in y_vals])

    fig = plt.figure(figsize=(10, 6))
    ax = fig.gca()

    # Shade the selected interval union in the displayed plotting window.
    used_interval_label = False
    for interval in intervals:
        left = float(interval["a"])
        right = y_max if interval["unbounded"] else min(float(interval["b"]), y_max)
        if right > 0.0 and left < y_max:
            ax.axvspan(
                max(left, 0.0),
                right,
                alpha=0.12,
                label="selected interval(s)" if not used_interval_label else None,
            )
            used_interval_label = True

    ax.plot(y_vals, profits, label="localized terminal financed profit Pi(S_T)")
    ax.axhline(0.0, linewidth=1.5, linestyle="--")

    target_points = np.asarray(focused_result["target_points"], dtype=float)
    target_points = target_points[target_points <= y_max]
    if len(target_points):
        target_profits = np.array([
            portfolio_terminal_profit(float(y), focused_result)
            for y in target_points
        ])
        ax.scatter(target_points, target_profits, zorder=3, label="target certificate points")

    ax.fill_between(y_vals, profits, 0.0, where=(profits > 0.0), alpha=0.25, label="positive")
    ax.fill_between(y_vals, profits, 0.0, where=(profits < 0.0), alpha=0.25, label="negative")

    ax.set_xlabel("Terminal stock price S_T")
    ax.set_ylabel("Terminal financed profit Pi(S_T)")
    ax.set_title("Localized deterministic arbitrage on selected intervals")
    ax.grid(True)
    ax.legend()
    plt.show()


def run_interactive_localized_interval_arbitrage(base_result, mode="paper_sum_D"):
    """Ask for intervals, solve the localized LP, print tables, and plot Pi."""
    intervals = _read_intervals_interactively()
    focused = build_and_solve_localized_interval_arbitrage(
        base_result,
        intervals,
        mode=mode,
    )
    print_localized_interval_arbitrage_report(focused)
    plot_localized_interval_profit_function(focused)
    return focused


# -----------------------------------------------------------------------------
# Start the interactive localized construction.
# -----------------------------------------------------------------------------
# Choose the localized objective.
#
#   "common_t"    : recommended if you want a direct certificate that the
#                   payoff is positive on every selected interval.
#   "paper_sum_D" : reproduces the old/direct Section-5 objective with one
#                   D_y variable per target certificate point.
#
LOCALIZED_LP_MODE = "paper_sum_D"

try:
    results
except NameError as exc:
    raise RuntimeError(
        "Run the previous analysis cell first so that the variable `results` exists."
    ) from exc

localized_results = run_interactive_localized_interval_arbitrage(
    results,
    mode=LOCALIZED_LP_MODE,
)



## Optional Cell 6 -- Run the interactive analysis with advanced settings

Use this cell instead of Cell 4 if you want to expose additional modelling parameters, such as the risk-free rate, use of `lastPrice`, synthetic stock spread, absolute perturbation, and setup-cost budget.

For the exact deterministic-arbitrage test in the paper, keep `setup_cost_budget = 0.0`.


In [ ]:

# Uncomment and run if you want advanced prompts instead of the standard prompts.
# results = run_interactive_analysis(ask_advanced_settings=True)



## Optional Cell 7 -- Audit the effective quotes

This table shows the bid and ask prices that were actually passed to the LP after cleaning and perturbation.


In [ ]:

# Uncomment after running the main analysis if you want to inspect the effective quotes.
# display_used_quotes(results["calls"], results["puts"], max_rows=80)
